# **The non-bonded potentials: van der Waals (Lennard-Jones) and electrostatics (Coulomb)**

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

Every other `LJ-ELEC_*` notebook in this repository **moves** particles — downhill (energy
minimisation), through time (molecular dynamics) or by chance (Monte Carlo). All of them are
driven by the *same* two pair potentials. This notebook stands still and looks at those two
functions themselves:

* the **van der Waals** interaction, modelled by the **Lennard-Jones** potential — and in
  particular its **repulsive** and **attractive** components, which we plot separately;
* the **electrostatic** interaction, modelled by the **Coulomb** potential.

No simulation, no sampling: just the energy of **one pair of atoms** as a function of their
separation $r$. We use **real force-field numbers** throughout — distances in ångström (Å),
energies in kcal/mol, charges in elementary charges ($e$) — so every value on these plots can be
compared with a force-field parameter file or a textbook.


---

## **1. The Lennard-Jones potential**

Two neutral atoms feel two opposing effects:

| | physical origin | how it decays | sign |
|---|---|---|---|
| **Repulsion** | **Pauli exclusion**: at short range the electron clouds overlap and the electrons cannot occupy the same states | very steep, modelled as $r^{-12}$ | $>0$ (pushes apart) |
| **Attraction** | **London dispersion**: an instantaneous dipole in one atom induces a dipole in the other | $r^{-6}$ — this exponent is *derived*, not fitted | $<0$ (pulls together) |

Adding them gives the **12-6 Lennard-Jones** potential:

$$\boxed{\;U_{LJ}(r)\;=\;\underbrace{4\epsilon\left(\frac{\sigma}{r}\right)^{12}}_{\text{repulsion}}\;-\;\underbrace{4\epsilon\left(\frac{\sigma}{r}\right)^{6}}_{\text{attraction}}\;=\;4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]\;}$$

The two parameters have a direct geometric meaning:

* $\sigma$ — the distance at which the energy **crosses zero**, i.e. where repulsion and
  attraction exactly cancel. Loosely, the atom's "diameter";
* $\epsilon$ — the **depth of the well**. The minimum $U_{LJ}=-\epsilon$ sits at
  $R_{min}=2^{1/6}\sigma\approx1.122\,\sigma$, slightly *outside* $\sigma$.

**The $r^{-12}$ term has no physical justification**: the true repulsion is closer to
exponential ($e^{-r/\rho}$, as in the Buckingham potential). Twelve is chosen because
$r^{-12}=\left(r^{-6}\right)^{2}$, so once the code has computed $(\sigma/r)^6$ the repulsion
costs one extra multiplication — a real consideration when this had to run on 1970s hardware,
and still the reason 12-6 is everywhere today.

### Two conventions — mind which one your force field uses

The formula above is the **$\sigma$ form** (OPLS, GROMACS topologies, most textbooks). AMBER and
CHARMM instead tabulate the position of the minimum, and write

$$U_{LJ}(r)=\epsilon\left[\left(\frac{R_{min}}{r}\right)^{12}-2\left(\frac{R_{min}}{r}\right)^{6}\right],
\qquad R_{min}=2^{1/6}\,\sigma$$

Both give **exactly the same curve** with the same $\epsilon$ — but $\sigma$ and $R_{min}$ differ
by 12 %, and the parameter files usually list $R_{min}/2$ (the atom's "radius"), not $R_{min}$.
Substituting one for the other is a classic and silent mistake.


---
## **2. The Coulomb potential**

Two charges $q_a$ and $q_b$ separated by $r$ interact with

$$U_{Coul}(r)\;=\;\frac{1}{4\pi\varepsilon_0}\,\frac{q_a q_b}{\varepsilon_r\, r}\;=\;332.06\;\frac{q_a q_b}{\varepsilon_r\, r}
\quad\text{(kcal/mol, with $q$ in $e$ and $r$ in Å)}$$

* **like charges** ($q_aq_b>0$): $U>0$ everywhere — pure repulsion;
* **unlike charges** ($q_aq_b<0$): $U<0$ everywhere — pure attraction, and it **diverges** as
  $r\to0$ (nothing stops the collapse except the Lennard-Jones wall — which is precisely why
  the MD notebook needs a [soft core](LJ-ELEC_MD-SoftCore.ipynb));
* the **relative dielectric constant** $\varepsilon_r$ *screens* the interaction:
  $\varepsilon_r=1$ in vacuum, $\approx 4$ inside a protein core, $\approx 80$ in bulk water.

The decisive difference from van der Waals is the **range**. Coulomb falls off as $r^{-1}$,
dispersion as $r^{-6}$. At ten atomic diameters the van der Waals attraction is a *millionth* of
its contact value and can be safely truncated; the Coulomb term is still a *tenth* of its contact
value — which is why the non-bonded cutoff used by every simulation is a far more serious
approximation for the charges than for the van der Waals term (§12).

---

## **Showing and hiding the code**

The Python here is a means to an end, so the **plotting cells start collapsed**: you see the
figure, not the forty lines of matplotlib that drew it. Nothing is disabled — a collapsed cell
runs exactly like an open one.

* **One cell:** click the blue **collapser bar** just left of it, or use *View → Collapse Selected
  Code*. Click again to bring it back.
* **All of them:** *View → Collapse All Code* / *Expand All Code*.

---
## **3. Imports and plotting palette**

Only **matplotlib** and **numpy** are needed (both already in `requirements.txt`); the cell
installs them if they are missing, which makes the notebook work as-is on Google Colab.

Throughout the notebook the colour code is fixed and meaningful:
**<span style="color:#d62728">red = repulsive</span>**,
**<span style="color:#1f77b4">blue = attractive</span>**,
**<span style="color:#6a3d9a">purple = the total</span>** (the sum of the two). The same three
colours are reused for the Coulomb curves, so a red curve always means "pushes apart" and a blue
curve always means "pulls together".

In [ ]:
# Install required packages if missing (e.g. on Google Colab)
import importlib.util, subprocess, sys

for pkg in ["matplotlib", "numpy"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import rc

rc('animation', html='jshtml')
%matplotlib inline

# --- fixed, colour-vision-safe palette: meaning is attached to the colour ---
C_REP = "#d62728"   # repulsive  contribution / like-charge pair
C_ATT = "#1f77b4"   # attractive contribution / unlike-charge pair
C_TOT = "#6a3d9a"   # the total (sum) curve
C_REF = "#8a8a8a"   # reference lines, axes, annotations

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2.0,
})

---
## **4. Parameters — a real carbon–carbon pair**

The Lennard-Jones parameters below are those of an **aliphatic sp³ carbon** (atom type `CT`) from
the **AMBER** ff14SB force field, for a C···C pair:

$$R_{min}/2 = 1.908\ \text{Å}\quad\Longrightarrow\quad R_{min}=3.816\ \text{Å},\qquad
\sigma = \frac{R_{min}}{2^{1/6}} = 3.400\ \text{Å},\qquad \epsilon = 0.1094\ \text{kcal/mol}$$

These are the numbers that hold two touching hydrocarbon chains together. They are worth
remembering as an order of magnitude: **a carbon atom is about 3.4 Å across, and one C···C van der
Waals contact is worth about a tenth of a kcal/mol.**

For the electrostatics we use two realistic charge scales:

* $q=\pm0.1\,e$ — a typical **partial charge** on an aliphatic carbon;
* $q=\pm1.0\,e$ — a **full ionic charge**, as on a carboxylate or an ammonium group (or Na⁺/Cl⁻).

| Parameter | Meaning | Value |
|---|---|---|
| `Sigma` | LJ distance parameter — where $U_{LJ}=0$ | 3.400 Å |
| `Rmin` | position of the LJ minimum, $2^{1/6}\sigma$ | 3.816 Å |
| `Epsilon` | LJ well depth | 0.1094 kcal/mol |
| `q_part`, `q_ion` | partial / full charge | 0.1 $e$, 1.0 $e$ |
| `Dielec` | relative dielectric constant $\varepsilon_r$ | 1 (vacuum) |
| `CutOff` | non-bonded cutoff | 10 Å (a typical MD value) |
| `Temperature` | temperature, for the thermal energy scale $k_BT$ | 300 K |
| `kB` | Boltzmann constant | 0.0019872 kcal/(mol·K) |
| `Mass` | atomic mass of carbon (used only for the collision in §14) | 12.011 amu |

> **Relation to the other notebooks.** The simulation notebooks in this repository use exactly
> the same formula and the same two symbols — `Sigma` for $\sigma$, `Epsilon` for $\epsilon$ —
> but in *toy* units: a 500×500 box, particle radius 25, `Sigma = 56`, `Epsilon = 6.25`. Only the
> numbers differ. Everything in *this* notebook is in real force-field units so that the values
> can be compared with the literature.

In [ ]:
# Lennard-Jones: aliphatic sp3 carbon (AMBER ff14SB type CT), a C...C pair
Epsilon = 0.1094          # kcal/mol   well depth
Sigma   = 3.400           # Angstrom   U_LJ(Sigma) = 0
Rmin    = 2**(1/6)*Sigma  # Angstrom   position of the minimum (AMBER's Rmin = 2 * 1.908)

# --- electrostatics ---
COULOMB_K = 332.0637      # kcal.A/(mol.e^2)  = 1/(4 pi eps0) in these units
q_part  = 0.1             # e   typical partial charge on an aliphatic carbon
q_ion   = 1.0             # e   full ionic charge (carboxylate, ammonium, Na+, Cl-)
Dielec  = 1.0             # relative dielectric constant (1 = vacuum)

# --- simulation-style settings ---
CutOff      = 10.0        # Angstrom   a typical non-bonded cutoff
Temperature = 300.0       # K
kB          = 0.0019872   # kcal/(mol.K)
kT          = kB*Temperature
Mass        = 12.011      # amu (carbon), used only for the collision in section 15

print(f"sigma  (U_LJ = 0)        : {Sigma:7.3f} A")
print(f"Rmin   (U_LJ minimal)    : {Rmin:7.3f} A   = 2^(1/6) * sigma")
print(f"epsilon (well depth)     : {Epsilon:7.4f} kcal/mol"
      f"  = {Epsilon*4.184:.4f} kJ/mol = {Epsilon/kB:.1f} K")
print(f"kT at {Temperature:.0f} K             : {kT:7.4f} kcal/mol")
print()
print(f"-> one C...C contact is only {Epsilon/kT:.2f} kT deep: on its own it cannot hold")
print(f"   two atoms together at room temperature. It takes MANY such contacts.")

---
## **5. The energy functions**

Written out term by term, exactly as in the formulas above. The Lennard-Jones energy is returned
**split into its repulsive and attractive parts** — that split is what this notebook is about,
and no simulation ever needs it.

In [ ]:
# Defining the Lennard-Jones and Coulomb functions
def lj_repulsion(r, epsilon=Epsilon, sigma=Sigma):
    '''Pauli repulsion: +4 eps (sigma/r)^12. Always positive, always pushes apart.'''
    return 4*epsilon * (sigma/np.asarray(r, dtype=float))**12

def lj_attraction(r, epsilon=Epsilon, sigma=Sigma):
    '''London dispersion: -4 eps (sigma/r)^6. Always negative, always pulls together.'''
    return -4*epsilon * (sigma/np.asarray(r, dtype=float))**6

def lj(r, epsilon=Epsilon, sigma=Sigma):
    '''Total Lennard-Jones energy = repulsion + attraction  [kcal/mol].'''
    return lj_repulsion(r, epsilon, sigma) + lj_attraction(r, epsilon, sigma)

def lj_force(r, epsilon=Epsilon, sigma=Sigma):
    '''Radial force -dU/dr  [kcal/(mol.A)]. Positive = repulsive (pushes the pair apart).'''
    r = np.asarray(r, dtype=float)
    return 24*epsilon/r * (2*(sigma/r)**12 - (sigma/r)**6)

# ---------- Coulomb ----------
def coulomb(r, qa, qb, dielec=Dielec):
    '''Coulomb energy  k qa qb / (eps_r r)  [kcal/mol], q in e and r in Angstrom.'''
    return COULOMB_K*qa*qb / (dielec*np.asarray(r, dtype=float))

def coulomb_force(r, qa, qb, dielec=Dielec):
    '''Radial Coulomb force -dU/dr = k qa qb / (eps_r r^2). Positive = repulsive.'''
    return COULOMB_K*qa*qb / (dielec*np.asarray(r, dtype=float)**2)

---
## **6. The Lennard-Jones potential and its two components**

This is the central figure of the notebook. The **red dashed** curve is the repulsive $r^{-12}$
term, the **blue dashed** curve the attractive $r^{-6}$ term, and the **purple solid** curve
their sum — the Lennard-Jones potential itself.

Read it from right to left, the way two approaching atoms experience it:

* **far away** both terms are essentially zero — the atoms do not feel each other;
* **approaching**, the attraction ($r^{-6}$, longer-ranged) switches on first and the total
  energy goes *negative*: the pair is bound. The shaded blue region is where attraction wins;
* at $r=R_{min}=2^{1/6}\sigma=3.82$ Å the two terms **balance in slope** and the curve reaches
  its minimum, $-\epsilon=-0.109$ kcal/mol. This is the equilibrium contact distance — the
  distance you would measure between two touching carbon atoms;
* **closer still**, the much steeper $r^{-12}$ term takes over: the total crosses zero exactly at
  $r=\sigma=3.40$ Å and then rises almost vertically. The shaded red region is where repulsion
  wins. Pushing two carbons to 3 Å already costs 1.04 kcal/mol — nearly *ten times* the depth of
  the well they just climbed out of, for a squeeze of less than 1 Å.

The right panel zooms into the well itself — the only part of the curve a simulation at ordinary
temperature actually samples.

In [ ]:
# Visualizing the Lennard-Jones potential

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12.5, 4.6))

# ---------- left: the full picture, components and total ----------
r = np.linspace(0.86*Sigma, 3.2*Sigma, 800)
ax0.plot(r, lj_repulsion(r), color=C_REP, ls="--", lw=1.7,
         label=r"repulsion  $+4\epsilon(\sigma/r)^{12}$")
ax0.plot(r, lj_attraction(r), color=C_ATT, ls="--", lw=1.7,
         label=r"attraction  $-4\epsilon(\sigma/r)^{6}$")
ax0.plot(r, lj(r), color=C_TOT, lw=2.4, label=r"total  $U_{LJ}$")
ax0.axhline(0, color=C_REF, lw=0.8)
ax0.set_ylim(-6*Epsilon, 9*Epsilon)
ax0.set_xlabel("pair separation  r  (Å)"); ax0.set_ylabel("energy  (kcal/mol)")
ax0.set_title("Lennard-Jones = repulsion + attraction")
ax0.legend(loc="upper right")

ax0.fill_between(r, 0, lj(r), where=lj(r) < 0, color=C_ATT, alpha=0.12, lw=0)
ax0.fill_between(r, 0, np.clip(lj(r), None, 9*Epsilon), where=lj(r) > 0,
                 color=C_REP, alpha=0.12, lw=0)
ax0.annotate("attraction wins", xy=(1.55*Sigma, -3.0*Epsilon), color=C_ATT, fontsize=9)
ax0.annotate("repulsion wins", xy=(0.96*Sigma, 1.4*Epsilon), xytext=(1.30*Sigma, 4.6*Epsilon),
             color=C_REP, fontsize=9, arrowprops=dict(arrowstyle="->", color=C_REP, lw=0.9))

# ---------- right: zoom on the well ----------
rz = np.linspace(0.94*Sigma, 2.3*Sigma, 800)
ax1.plot(rz, lj(rz), color=C_TOT, lw=2.4)
ax1.fill_between(rz, 0, lj(rz), where=lj(rz) < 0, color=C_ATT, alpha=0.15, lw=0)
ax1.fill_between(rz, 0, np.clip(lj(rz), None, 1.6*Epsilon), where=lj(rz) > 0,
                 color=C_REP, alpha=0.15, lw=0)
ax1.axhline(0, color=C_REF, lw=0.8)
ax1.set_ylim(-1.8*Epsilon, 1.6*Epsilon)
ax1.set_xlabel("pair separation  r  (Å)"); ax1.set_ylabel("energy  (kcal/mol)")
ax1.set_title(r"the well: minimum $-\epsilon$ at $r=R_{min}=2^{1/6}\sigma$")

ax1.plot([Sigma], [0.0], "o", ms=7, color=C_REF, zorder=5)
ax1.annotate(rf"$\sigma$ = {Sigma:.2f} Å   ($U=0$)", xy=(Sigma, 0),
             xytext=(Sigma+0.35, 0.75*Epsilon), color="#444444", fontsize=9,
             arrowprops=dict(arrowstyle="-", color=C_REF, lw=0.8))
ax1.plot([Rmin], [-Epsilon], "o", ms=8, color=C_TOT, zorder=5)
ax1.annotate(rf"$R_{{min}}$ = {Rmin:.2f} Å,  $U=-\epsilon$ = {-Epsilon:.4f}",
             xy=(Rmin, -Epsilon), xytext=(Rmin+0.75, -1.45*Epsilon),
             color=C_TOT, fontsize=9, arrowprops=dict(arrowstyle="-", color=C_TOT, lw=0.8))

plt.tight_layout(); plt.show()

print(f"At r = sigma = {Sigma:.2f} A the two components cancel exactly:")
print(f"   repulsion  = {float(lj_repulsion(Sigma)):+8.4f} kcal/mol")
print(f"   attraction = {float(lj_attraction(Sigma)):+8.4f} kcal/mol")
print(f"   total      = {float(lj(Sigma)):+8.4f} kcal/mol")
print(f"\nCost of squeezing the pair to 3.0 A: {float(lj(3.0)):.3f} kcal/mol "
      f"= {float(lj(3.0))/kT:.1f} kT")

---
## **7. What the two parameters do — and how real atoms differ**

$\epsilon$ and $\sigma$ have completely separate jobs:

* **$\epsilon$ scales the curve vertically.** It multiplies *both* components equally, so the
  shape and the position of the minimum are untouched — only the depth changes, linearly.
* **$\sigma$ stretches the curve horizontally.** It sets where the wall stands, and hence the
  size of the atom. The depth is unchanged; everything simply moves out.

This is why force-field developers can fit the two parameters almost independently. The third
panel uses **real parameters for four common atom types** (AMBER ff14SB, like-pair interactions),
so you can see the actual spread:

| atom type | $R_{min}/2$ (Å) | $\sigma$ (Å) | $\epsilon$ (kcal/mol) |
|---|---|---|---|
| H (aliphatic, `HC`) | 1.487 | 2.650 | 0.0157 |
| C (aliphatic, `CT`) | 1.908 | 3.400 | 0.1094 |
| O (carbonyl, `O`)   | 1.661 | 2.960 | 0.2100 |
| S (thiol/thioether, `SH`) | 2.000 | 3.564 | 0.2500 |

Note that oxygen is **smaller** than carbon yet has a **twice deeper** well — size and stickiness
really are independent knobs. And note the grey band: *every one of these wells is well under
$k_BT$*.

In [ ]:
# What do epsilon and sigma do
# --- real AMBER ff14SB parameters: (name, Rmin/2 [A], epsilon [kcal/mol]) ---
ATOM_TYPES = [("H  (HC)", 1.487, 0.0157),
              ("C  (CT)", 1.908, 0.1094),
              ("O  (O) ", 1.661, 0.2100),
              ("S  (SH)", 2.000, 0.2500)]

fig, (axE, axS, axR) = plt.subplots(1, 3, figsize=(15.5, 4.4))
blues = plt.cm.Blues(np.linspace(0.45, 0.97, 4))

# ---- vary epsilon at fixed sigma ----
r = np.linspace(0.95*Sigma, 3.0*Sigma, 600)
for c, eps in zip(blues, [0.05, 0.1094, 0.20, 0.35]):
    axE.plot(r, lj(r, epsilon=eps), color=c, label=rf"$\epsilon$ = {eps:.4g}")
    axE.plot([Rmin], [-eps], "o", ms=5, color=c)
axE.axvline(Rmin, color=C_REF, ls=":", lw=0.9)
axE.annotate(r"$R_{min}$ does not move", xy=(Rmin, 0.20), xytext=(Rmin+0.35, 0.24),
             color="#444444", fontsize=9)
axE.axhline(0, color=C_REF, lw=0.8); axE.set_ylim(-0.45, 0.32)
axE.set_xlabel("r  (Å)"); axE.set_ylabel("energy  (kcal/mol)")
axE.set_title(r"$\epsilon$ sets the DEPTH"); axE.legend(loc="upper right")

# ---- vary sigma at fixed epsilon ----
r2 = np.linspace(0.75*Sigma, 4.0*Sigma, 800)
for c, sg in zip(blues, [2.6, 3.4, 4.2, 5.0]):
    axS.plot(r2, lj(r2, sigma=sg), color=c, label=rf"$\sigma$ = {sg:.1f} Å")
    axS.plot([2**(1/6)*sg], [-Epsilon], "o", ms=5, color=c)
axS.axhline(-Epsilon, color=C_REF, ls=":", lw=0.9)
axS.annotate(r"depth stays $-\epsilon$", xy=(11.5, -Epsilon), xytext=(9.0, -Epsilon+0.035),
             color="#444444", fontsize=9)
axS.axhline(0, color=C_REF, lw=0.8); axS.set_ylim(-0.22, 0.25)
axS.set_xlabel("r  (Å)"); axS.set_ylabel("energy  (kcal/mol)")
axS.set_title(r"$\sigma$ sets the SIZE"); axS.legend(loc="upper right")

# ---- real atom types ----
r3 = np.linspace(2.0, 11.0, 800)
for c, (name, rmin_half, eps) in zip(blues, ATOM_TYPES):
    sg = 2*rmin_half / 2**(1/6)
    axR.plot(r3, lj(r3, epsilon=eps, sigma=sg), color=c, label=name)
    axR.plot([2*rmin_half], [-eps], "o", ms=5, color=c)
axR.axhline(0, color=C_REF, lw=0.8)
axR.axhspan(-kT, kT, color=C_REF, alpha=0.10, lw=0)
axR.axhline(kT, color=C_REF, ls="--", lw=0.9); axR.axhline(-kT, color=C_REF, ls="--", lw=0.9)
axR.annotate(rf"$\pm k_BT$ ({Temperature:.0f} K)", xy=(9.0, kT), xytext=(5.4, 1.04*kT),
             color="#444444", fontsize=9)
axR.set_ylim(-0.30, 0.78)
axR.set_xlabel("r  (Å)"); axR.set_ylabel("energy  (kcal/mol)")
axR.set_title("real atom types (AMBER ff14SB)"); axR.legend(loc="upper right")

plt.tight_layout(); plt.show()

print(f"{'atom type':>10} {'sigma (A)':>10} {'Rmin (A)':>9} {'eps (kcal/mol)':>15} {'well in kT':>11}")
for name, rmin_half, eps in ATOM_TYPES:
    print(f"{name:>10} {2*rmin_half/2**(1/6):10.3f} {2*rmin_half:9.3f} {eps:15.4f} {eps/kT:11.2f}")
print("\nEvery one of them is well under 1 kT: a SINGLE van der Waals contact is always weak.")

---
## **8. From energy to force**

A simulation does not move atoms with the energy — it moves them with the **force**, the negative
slope of the energy:

$$F(r)\;=\;-\frac{dU_{LJ}}{dr}\;=\;\frac{24\epsilon}{r}\left[2\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]$$

Three things to read off the right panel:

* the force is **zero exactly at the energy minimum** $R_{min}$ — the definition of equilibrium;
* **inside** $R_{min}$ the force is positive (repulsive) and rises like $r^{-13}$ — even steeper
  than the energy. This is the divergence the [soft core](LJ-ELEC_MD-SoftCore.ipynb) exists to
  tame;
* **outside** $R_{min}$ the force is negative (attractive) and reaches its most negative value at
  $r=(26/7)^{1/6}\sigma\approx1.24\,\sigma = 4.23$ Å: that is the **strongest pull** two carbons
  ever exert on each other, and it is only $0.077$ kcal/(mol·Å) — about 5 pN, a thousand times
  weaker than the force needed to break a covalent bond. Beyond it the attraction fades as
  $r^{-7}$.

In [ ]:
# Plot the Lennard Jones potential and the corresponding force function
fig, (axU, axF) = plt.subplots(1, 2, figsize=(12.5, 4.4), sharex=True)

r = np.linspace(0.96*Sigma, 2.4*Sigma, 900)
U, F = lj(r), lj_force(r)

axU.plot(r, U, color=C_TOT, lw=2.3)
axU.axhline(0, color=C_REF, lw=0.8)
axU.axvline(Rmin, color=C_REF, ls=":", lw=1.0)
axU.plot([Rmin], [-Epsilon], "o", ms=8, color=C_TOT)
axU.annotate("minimum", xy=(Rmin, -Epsilon), xytext=(Rmin+0.8, -Epsilon-0.035),
             color=C_TOT, fontsize=9, arrowprops=dict(arrowstyle="-", color=C_TOT, lw=0.8))
axU.set_ylim(-0.20, 0.30)
axU.set_ylabel("energy  $U_{LJ}(r)$   (kcal/mol)"); axU.set_xlabel("pair separation  r  (Å)")
axU.set_title("energy")

axF.plot(r, F, color=C_TOT, lw=2.3)
axF.fill_between(r, 0, F, where=F > 0, color=C_REP, alpha=0.15, lw=0)
axF.fill_between(r, 0, F, where=F < 0, color=C_ATT, alpha=0.15, lw=0)
axF.axhline(0, color=C_REF, lw=0.8)
axF.axvline(Rmin, color=C_REF, ls=":", lw=1.0)
axF.plot([Rmin], [0.0], "o", ms=8, color=C_TOT)
axF.annotate(rf"$F=0$ at $R_{{min}}$ = {Rmin:.2f} Å", xy=(Rmin, 0),
             xytext=(Rmin+0.8, 0.45), color="#444444", fontsize=9,
             arrowprops=dict(arrowstyle="-", color=C_REF, lw=0.8))

r_pull = (26/7)**(1/6) * Sigma                    # where the attractive force is strongest
axF.plot([r_pull], [lj_force(r_pull)], "o", ms=7, color=C_ATT)
axF.annotate(f"strongest pull:  r = {r_pull:.2f} Å,  F = {float(lj_force(r_pull)):.3f}",
             xy=(r_pull, float(lj_force(r_pull))), xytext=(r_pull+0.5, -0.16),
             color=C_ATT, fontsize=9, arrowprops=dict(arrowstyle="-", color=C_ATT, lw=0.8))
axF.annotate("pushes apart", xy=(3.35, 0.85), color=C_REP, fontsize=9)
axF.annotate("pulls together", xy=(6.0, -0.06), color=C_ATT, fontsize=9)
axF.set_ylim(-0.22, 1.15)     # clipped: the repulsive branch runs off the top, diverging as r^-13
axF.set_ylabel("radial force  $-dU/dr$   (kcal/(mol·Å))"); axF.set_xlabel("pair separation  r  (Å)")
axF.set_title("force  (+ = repulsive,  $-$ = attractive)")

plt.tight_layout(); plt.show()

# numerical check that F really is -dU/dr
h = 1e-6
r_chk = np.array([0.98*Sigma, Rmin, 1.3*Sigma, 2.0*Sigma])
num = -(lj(r_chk+h) - lj(r_chk-h)) / (2*h)
print("    r (A)     F analytic      F numeric")
for rc, fa, fn in zip(r_chk, lj_force(r_chk), num):
    print(f"{rc:9.3f} {fa:14.7f} {fn:14.7f}")
print(f"\nstrongest attractive force = {float(lj_force(r_pull)):.4f} kcal/(mol.A) "
      f"= {abs(float(lj_force(r_pull)))*69.48:.1f} pN")

---
## **9. Is the well deep enough to matter? Comparing with $k_BT$**

An interaction only *survives* if it is not washed out by thermal motion, so the meaningful
yardstick for a well depth is $k_BT$, not zero. Here the real numbers deliver the single most
important message of this notebook:

$$\epsilon = 0.109\ \text{kcal/mol},\qquad k_BT(300\,\text{K}) = 0.596\ \text{kcal/mol}
\qquad\Longrightarrow\qquad \epsilon \approx 0.18\,k_BT$$

**One carbon–carbon van der Waals contact is roughly five times weaker than thermal energy.** On
its own it holds nothing together. The right panel says the same thing probabilistically: in the
dilute limit the chance of finding the pair at separation $r$ is the **Boltzmann factor**
$e^{-U(r)/k_BT}$, and at 300 K it peaks at a mere **1.20** — the atoms are only 20 % more likely
to be in contact than anywhere else. You have to cool to 100 K before the preference becomes
pronounced.

So why do apolar molecules stick together at all? Because van der Waals contacts come in **large
numbers** — a protein core or a lipid bilayer makes hundreds of them at once, and hundreds of
$0.18\,k_BT$ add up. (In water the *hydrophobic effect* adds a further, entropic driving force
that this pair potential does not describe at all.) The repulsive wall, by contrast, is
absolutely impenetrable at *every* temperature: no thermal energy available at 300 K comes close
to the cost of overlapping two atoms.

In [ ]:
# Plot the Lennard-Jones potential and compare with kT
fig, (axK, axB) = plt.subplots(1, 2, figsize=(12.5, 4.4))

r = np.linspace(0.95*Sigma, 3.0*Sigma, 700)
axK.plot(r, lj(r), color=C_TOT, lw=2.3)
axK.axhline(0, color=C_REF, lw=0.8)
axK.axhspan(-kT, kT, color=C_REF, alpha=0.18, lw=0)
axK.annotate(rf"$\pm k_BT$ = {kT:.3f} kcal/mol  ($T$ = {Temperature:.0f} K)",
             xy=(6.5, kT), xytext=(4.6, 1.12*kT), color="#444444", fontsize=9)
axK.plot([Rmin], [-Epsilon], "o", ms=8, color=C_TOT)
axK.annotate(rf"well = only {Epsilon/kT:.2f} $k_BT$", xy=(Rmin, -Epsilon),
             xytext=(Rmin+0.7, -Epsilon-0.28), color=C_TOT, fontsize=9,
             arrowprops=dict(arrowstyle="-", color=C_TOT, lw=0.8))
axK.set_ylim(-0.85, 0.95)
axK.set_xlabel("pair separation  r  (Å)"); axK.set_ylabel("energy  (kcal/mol)")
axK.set_title("the C···C well is small compared with thermal energy")

purples = plt.cm.Purples(np.linspace(0.50, 0.97, 4))
for c, T in zip(purples, [1000.0, 600.0, 300.0, 100.0]):
    axB.plot(r, np.exp(-lj(r)/(kB*T)), color=c, label=f"T = {T:.0f} K")
axB.axhline(1.0, color=C_REF, ls="--", lw=0.9)
axB.annotate("indifferent", xy=(9.0, 1.0), xytext=(8.0, 1.10), color="#444444", fontsize=9)
axB.axvline(Rmin, color=C_REF, ls=":", lw=0.9)
axB.annotate(r"$R_{min}$", xy=(Rmin, 0.15), xytext=(Rmin+0.2, 0.10), color="#444444", fontsize=9)
axB.set_ylim(0, 2.0)
axB.set_xlabel("pair separation  r  (Å)"); axB.set_ylabel(r"$e^{-U_{LJ}(r)/k_BT}$")
axB.set_title("Boltzmann factor: how likely is that separation?")
axB.legend(loc="upper right")

plt.tight_layout(); plt.show()

print(f"{'T (K)':>7} {'kT (kcal/mol)':>15} {'eps/kT':>9} {'peak e^(eps/kT)':>17}")
for T in (1000, 600, 300, 100):
    print(f"{T:7.0f} {kB*T:15.4f} {Epsilon/(kB*T):9.3f} {math.exp(Epsilon/(kB*T)):17.3f}")

---
## **10. The Coulomb potential**

Now the second half of the energy function. Compared with Lennard-Jones the Coulomb potential is
almost embarrassingly simple — a single $1/r$ — but it behaves completely differently:

* it has **no minimum and no length scale**. There is nothing in $q_aq_b/\varepsilon_r r$ that
  sets a preferred distance: like charges want to be infinitely far apart, unlike charges want to
  be on top of each other. Any structure in a charged system comes from the *competition* with
  the Lennard-Jones wall (§13);
* its **sign is set by the charges**, not by the distance;
* it is **enormous at short range**. A full ion pair ($\pm1\,e$) at van der Waals contact
  (3.82 Å) is worth $-87$ kcal/mol in vacuum — that is **800 times** the C···C van der Waals
  well, and 146 $k_BT$. This is why electrostatics, not dispersion, decides the structure of
  anything charged.

The right panel shows the rescue: the **dielectric constant divides the whole curve**. Going from
vacuum to water ($\varepsilon_r=80$) flattens it by a factor of 80 and brings a contact ion pair
down to $-1.09$ kcal/mol $\approx1.8\,k_BT$ — which is about the strength a **salt bridge** is
usually quoted to have on a protein surface. Screening is the reason biology can make and break
electrostatic contacts at all.

The two grey reference lines make the comparison concrete: the upper one is $k_BT$, the lower one
the whole C···C van der Waals well depth $\epsilon$. Note where each curve crosses them — in
vacuum the ion pair is still above $k_BT$ at 50 Å.

In [ ]:
# Visualise the Coulomb potential
fig, (axQ, axD) = plt.subplots(1, 2, figsize=(12.5, 4.4))

r = np.linspace(2.6, 12.0, 700)

# ---- like vs unlike full charges in vacuum ----
axQ.plot(r, coulomb(r, +q_ion, +q_ion, 1.0), color=C_REP, lw=2.3,
         label=r"like charges  $+1/+1$  (repulsive)")
axQ.plot(r, coulomb(r, +q_ion, -q_ion, 1.0), color=C_ATT, lw=2.3,
         label=r"unlike charges  $+1/-1$  (attractive)")
axQ.fill_between(r, 0, coulomb(r, +q_ion, +q_ion, 1.0), color=C_REP, alpha=0.10, lw=0)
axQ.fill_between(r, 0, coulomb(r, +q_ion, -q_ion, 1.0), color=C_ATT, alpha=0.10, lw=0)
axQ.axhline(0, color=C_REF, lw=0.8)
axQ.axvline(Rmin, color=C_REF, ls=":", lw=0.9)
e_contact = float(coulomb(Rmin, +q_ion, -q_ion, 1.0))
axQ.plot([Rmin], [e_contact], "o", ms=8, color=C_ATT)
axQ.annotate(f"at vdW contact ({Rmin:.2f} Å):\n{e_contact:.1f} kcal/mol = {abs(e_contact)/kT:.0f} $k_BT$",
             xy=(Rmin, e_contact), xytext=(Rmin+2.1, e_contact+52),
             color=C_ATT, fontsize=9, arrowprops=dict(arrowstyle="-", color=C_ATT, lw=0.8))
axQ.set_ylim(-140, 140)
axQ.set_xlabel("pair separation  r  (Å)"); axQ.set_ylabel("energy  (kcal/mol)")
axQ.set_title(r"a full ion pair in vacuum  ($q=\pm1\,e$, $\varepsilon_r$ = 1)")
axQ.legend(loc="upper right")

# ---- dielectric screening, on a log scale so all four fit ----
rd = np.linspace(2.6, 50.0, 700)
blues = plt.cm.Blues(np.linspace(0.97, 0.50, 4))
for c, d in zip(blues, [1.0, 4.0, 20.0, 80.0]):
    axD.loglog(rd, np.abs(coulomb(rd, +q_ion, -q_ion, d)), color=c, label=rf"$\varepsilon_r$ = {d:.0f}")
axD.axhline(kT, color=C_REF, ls="--", lw=1.1)
axD.annotate(rf"$k_BT$ = {kT:.2f}", xy=(40, kT), xytext=(20, 1.3*kT), color="#444444", fontsize=9)
axD.axhline(Epsilon, color=C_REF, ls=":", lw=1.1)
axD.annotate(rf"one C···C vdW well = {Epsilon:.3f}", xy=(40, Epsilon),
             xytext=(3.0, 0.35*Epsilon), color="#444444", fontsize=9)
axD.set_ylim(1e-2, 3e2)
axD.set_xticks([3, 5, 10, 20, 50]); axD.set_xticklabels(["3", "5", "10", "20", "50"])
axD.minorticks_off()
axD.set_xlabel("pair separation  r  (Å, log)"); axD.set_ylabel("|energy|  (kcal/mol, log)")
axD.set_title("screening: vacuum (1) → protein core (4) → water (80)")
axD.legend(loc="upper right")

plt.tight_layout(); plt.show()

print(f"Coulomb energy at the van der Waals contact distance r = {Rmin:.2f} A")
print(f"(for reference, the whole C...C vdW well is {-Epsilon:.4f} kcal/mol = {Epsilon/kT:.2f} kT)\n")
print(f"{'q (e)':>7} {'eps_r':>7} {'E (kcal/mol)':>14} {'in kT':>10} {'x vdW well':>12}")
for q in (q_part, 0.5, q_ion):
    for d in (1.0, 4.0, 80.0):
        e = float(coulomb(Rmin, +q, -q, d))
        print(f"{q:7.1f} {d:7.0f} {e:14.4f} {abs(e)/kT:10.2f} {abs(e)/Epsilon:12.1f}")

---
## **11. How fast do the van der Waals and electrostatic energies die away?**

Put the two attractive interactions on the same log-log axes and the gap is enormous. Both are
normalised here to their value at the van der Waals contact distance $R_{min}$, so both curves
start at 1 and the only thing being compared is **how fast they die**.

The dashed vertical line is a **10 Å cutoff**, a typical value in a real MD simulation: beyond it,
pairs are simply skipped. For the van der Waals term that is essentially exact — only **0.31 %**
of the contact attraction survives there, $-7\times10^{-4}$ kcal/mol, about a *thousandth* of
$k_BT$. Throwing it away changes nothing.

For the Coulomb term it is **not**. At 10 Å an ion pair still retains **38 %** of its contact
interaction: 33 kcal/mol in vacuum, and even in water ($\varepsilon_r=80$) 0.42 kcal/mol
$\approx0.7\,k_BT$ — and a plain cutoff truncates that abruptly to zero, putting an artificial
step in the energy and a spike in the force.

This is the practical reason real molecular-simulation codes do not treat electrostatics with a
plain cutoff, but with switching functions, reaction-field corrections or lattice-sum methods
(**Ewald / PME**). The toy simulations in this repository just truncate — fine for teaching, but
worth knowing.

In [ ]:
# Visualise the normalised energies as a function of distance
fig, ax = plt.subplots(figsize=(8.6, 4.8))

r = np.linspace(Rmin, 40.0, 900)
vdw_rel  = np.abs(lj_attraction(r)) / abs(float(lj_attraction(Rmin)))
coul_rel = np.abs(coulomb(r, q_ion, -q_ion, 1.0)) / abs(float(coulomb(Rmin, q_ion, -q_ion, 1.0)))

ax.loglog(r, vdw_rel,  color=C_ATT, lw=2.3, label=r"van der Waals attraction  $\propto r^{-6}$")
ax.loglog(r, coul_rel, color=C_REP, lw=2.3, label=r"Coulomb  $\propto r^{-1}$")
ax.axvline(CutOff, color=C_REF, ls="--", lw=1.2)
ax.annotate(f"cutoff = {CutOff:.0f} Å", xy=(CutOff, 2e-5), xytext=(CutOff*1.06, 2e-5),
            color="#444444", fontsize=9)

i_cut = int(np.argmin(np.abs(r-CutOff)))
v_at_cut, c_at_cut = float(vdw_rel[i_cut]), float(coul_rel[i_cut])
ax.plot([CutOff], [v_at_cut], "o", ms=8, color=C_ATT)
ax.plot([CutOff], [c_at_cut], "o", ms=8, color=C_REP)
ax.annotate(f"{c_at_cut*100:.0f} % left", xy=(CutOff, c_at_cut), xytext=(CutOff*1.15, c_at_cut*1.7),
            color=C_REP, fontsize=9)
ax.annotate(f"{v_at_cut*100:.2f} % left", xy=(CutOff, v_at_cut), xytext=(CutOff*0.36, v_at_cut*0.18),
            color=C_ATT, fontsize=9, arrowprops=dict(arrowstyle="-", color=C_ATT, lw=0.8))

ax.set_xlabel("pair separation  r  (Å, log)")
ax.set_ylabel("energy relative to its value at contact (log)")
ax.set_title("how fast the two interactions die away")
ax.set_ylim(1e-6, 2); ax.legend(loc="lower left")
ax.set_xticks([4, 5, 7, 10, 15, 20, 30, 40])
ax.set_xticklabels(["4", "5", "7", "10", "15", "20", "30", "40"])
ax.minorticks_off()
plt.tight_layout(); plt.show()

print(f"absolute energies at a {CutOff:.0f} A cutoff   (kT = {kT:.3f} kcal/mol):")
print(f"   van der Waals attraction, C...C : {float(lj_attraction(CutOff)):11.6f} kcal/mol "
      f"= {abs(float(lj_attraction(CutOff)))/kT:9.5f} kT")
for d in (1.0, 80.0):
    e = float(coulomb(CutOff, q_ion, -q_ion, d))
    print(f"   Coulomb +-1e, eps_r = {d:4.0f}          : {e:11.6f} kcal/mol "
          f"= {abs(e)/kT:9.5f} kT   <- truncated abruptly to 0")

---
## **12. Putting them together: the pair potential a simulation actually feels**

$$U(r)\;=\;U_{LJ}(r)\;+\;U_{Coul}(r)\;=\;4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]\;+\;332.06\,\frac{q_aq_b}{\varepsilon_r r}$$

This is the complete non-bonded energy every `LJ-ELEC_*` notebook evaluates. Both panels below use
the same two carbon atoms, now carrying a realistic **partial** charge of $q=\pm0.1\,e$ — the kind
an aliphatic carbon actually has. **Note the very different $y$-scales of the two panels.**

* **In vacuum** ($\varepsilon_r=1$, left) even that small partial charge dominates. The unlike
  pair gets a well **nine times deeper** than the van der Waals one ($-1.01$ instead of $-0.109$
  kcal/mol) and pulled inwards to 3.58 Å, and the like pair loses its minimum **entirely** — its
  energy just decreases all the way out, so the two atoms simply drift apart. A tenth of an
  electron charge is enough to erase a van der Waals well.
* **In water** ($\varepsilon_r=80$, right — note the roughly 10× finer scale) the same charges are
  screened into near-irrelevance. Both curves sit almost on top of the uncharged reference: the
  unlike well deepens by about 10 %, the like well shallows by about 10 %, and the *van der Waals*
  term is back in charge of the geometry.

That contrast is the whole story of solvent screening in one figure, and it is why a vacuum
simulation of a charged molecule behaves nothing like the same molecule in water. The dashed grey
curve in both panels is the **uncharged** Lennard-Jones reference.

In [ ]:
# Visualise the combined van der Waals and electrostatic potentials
def u_total(r, qa, qb, dielec):
    return lj(r) + coulomb(r, qa, qb, dielec)

def contact_minimum(qa, qb, dielec, lo=0.55, hi=4.0):
    '''Minimum of the total pair potential near contact; (None, None) if there is none.'''
    g = np.linspace(lo*Sigma, hi*Sigma, 400_001)
    u = u_total(g, qa, qb, dielec)
    i = int(np.argmin(u))
    if i in (0, len(g)-1):            # no turning point inside the window: monotonic
        return None, None
    return float(g[i]), float(u[i])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
r = np.linspace(0.88*Sigma, 3.4*Sigma, 900)

for ax, d, ylim, note in [(axes[0], 1.0,  (-1.35, 1.5),  "vacuum: the partial charges take over"),
                          (axes[1], 80.0, (-0.17, 0.19), "water: the charges are screened away")]:
    ax.plot(r, lj(r), color=C_REF, ls="--", lw=1.5, label="Lennard-Jones only (uncharged)")
    ax.plot(r, u_total(r, +q_part, -q_part, d), color=C_ATT, lw=2.3, label=r"unlike  $+0.1/-0.1\,e$")
    ax.plot(r, u_total(r, +q_part, +q_part, d), color=C_REP, lw=2.3, label=r"like  $+0.1/+0.1\,e$")
    ax.axhline(0, color=C_REF, lw=0.8)
    ax.axvline(Rmin, color=C_REF, ls=":", lw=0.9)

    for k, (label, (qa, qb), col) in enumerate([("unlike", (+q_part, -q_part), C_ATT),
                                                ("like  ", (+q_part, +q_part), C_REP)]):
        rm, um = contact_minimum(qa, qb, d)
        if rm is None:
            txt = f"{label} : no minimum — the energy falls off monotonically"
        else:
            ax.plot([rm], [um], "o", ms=8, color=col)
            txt = f"{label} : r = {rm:.2f} Å,  U = {um:+.3f} kcal/mol  ({um/kT:+.2f} $k_BT$)"
        ax.text(0.33, 0.135 - 0.075*k, txt, transform=ax.transAxes, color=col, fontsize=8.5)

    ax.set_ylim(*ylim)
    ax.set_xlabel("pair separation  r  (Å)"); ax.set_ylabel("total pair energy  (kcal/mol)")
    ax.set_title(rf"$\varepsilon_r$ = {d:.0f}   ({note})")

axes[0].legend(loc="upper right")
plt.tight_layout(); plt.show()


---
## **13. Watch an atom pair explore the curve**

Finally, the curves in motion. Two **carbon atoms** are aimed straight at each other and set
moving at about **16 Å/ps** — a typical speed for a carbon atom at 300 K — and we simply follow
where they sit on the Lennard-Jones curve as they approach, meet, and separate again.

What to watch:

1. **far apart** the curve is flat: the atoms barely feel each other and drift in at a steady
   speed;
2. **entering the well** the curve dips and the attraction pulls the pair together, so the atoms
   speed up a little. The dip is shallow — only 0.109 kcal/mol deep (§10) — so the effect is
   small: this pair is moving far too fast to be caught by it;
3. **meeting the wall** the repulsion takes over and the atoms slow sharply, stop, and reverse.
   That closest separation is marked in red: at this speed it is **2.90 Å**, so the pair pushes a
   long way *past* $\sigma=3.40$ Å and climbs well up the repulsive wall. Aim them faster and they
   stop further up the wall; aim them more gently and they turn around out near the bottom of the
   well;
4. **bouncing back** out again, symmetrically. The whole encounter is over in well under a
   picosecond.

Notice how little of the right-hand panel the *well* occupies. The vertical scale is set by how
far the pair climbs the repulsive wall (about 1.8 kcal/mol), and next to that the 0.109 kcal/mol
attractive well is barely a scratch. That is §10's message drawn to scale: the wall is what an
atom really feels, and a single van der Waals attraction is almost nothing in comparison.

*(How the positions are advanced step by step, using the force of §9, is the subject of the
[molecular-dynamics notebooks](LJ-ELEC_MD-Verlet.ipynb). Here we only use the result.)*

In [ ]:
# Visualize two atoms approaching each other
# --- unit bookkeeping: with A, amu and kcal/mol, the natural time unit is 48.888 fs ---
TIME_FS = 48.888                      # femtoseconds per internal time unit

# --- two carbon atoms aimed straight at each other, advanced in small steps ---
mu     = Mass/2.0                     # reduced mass of the relative coordinate (amu)
Speed  = 15.8                         # A/ps -- a typical speed for a carbon atom at 300 K
r0     = 4.2*Sigma                    # start far apart (A)
v0     = -Speed*TIME_FS/1000          # inward, in A per internal time unit
dt, nsteps = 0.002, 20000             # 0.1 fs per step, ~2 ps in total

r_t, v_t = r0, v0
a_t = float(lj_force(r_t))/mu         # the force of section 9, divided by the mass
traj_r, traj_v = [], []
for step in range(nsteps):
    r_new = r_t + v_t*dt + 0.5*a_t*dt*dt
    a_new = float(lj_force(r_new))/mu
    v_t   = v_t + 0.5*(a_t + a_new)*dt
    r_t, a_t = r_new, a_new
    traj_r.append(r_t)
    traj_v.append(abs(v_t)*1000/TIME_FS)          # speed in A/ps

traj_r, traj_v = np.array(traj_r), np.array(traj_v)
i_turn = int(np.argmin(traj_r)); r_turn = float(traj_r[i_turn])
print(f"approach speed        : {Speed:.1f} A/ps")
print(f"closest approach      : r = {r_turn:.2f} A   (sigma = {Sigma:.2f} A -> well up the wall)")
print(f"speed at that moment  : {traj_v[i_turn]:.2f} A/ps   (at rest for an instant, then pushed back out)")
print(f"fastest speed reached : {traj_v.max():.1f} A/ps   (in the well, where the attraction pulls hardest)")
print(f"time covered          : {nsteps*dt*TIME_FS/1000:.2f} ps")

# --- animation: the two atoms (left) and their position on the curve (right) ---
stride = max(1, nsteps//160)
frames = range(0, nsteps, stride)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.2, 4.6),
                               gridspec_kw={"width_ratios": [1.05, 1]})

axL.set_xlim(-2.4*Sigma, 2.4*Sigma); axL.set_ylim(-1.25*Sigma, 1.25*Sigma)
axL.set_aspect("equal"); axL.grid(False)
axL.set_xlabel("x  (Å)"); axL.set_title("two carbon atoms")
pA = plt.Circle((0, 0), Sigma/2, facecolor="#e8e4f0", edgecolor="k", lw=1.2)
pB = plt.Circle((0, 0), Sigma/2, facecolor="#e8e4f0", edgecolor="k", lw=1.2)
axL.add_patch(pA); axL.add_patch(pB)
txt = axL.text(0.02, 0.97, "", transform=axL.transAxes, fontsize=8.5, va="top",
               family="monospace")

rg = np.linspace(0.80*Sigma, 4.4*Sigma, 700)
axR.plot(rg, lj(rg), color=C_TOT, lw=2.2)
axR.axhline(0, color=C_REF, lw=0.8)
axR.axvline(r_turn, color=C_REP, ls=":", lw=1.1)
axR.annotate(f"closest approach:  r = {r_turn:.2f} Å", xy=(r_turn, 1.75),
             xytext=(r_turn+1.5, 1.95), color=C_REP, fontsize=9,
             arrowprops=dict(arrowstyle="->", color=C_REP, lw=0.9))
dot, = axR.plot([], [], "o", ms=11, color=C_TOT, zorder=5)
axR.set_ylim(-0.5, 2.5)
axR.set_xlabel("pair separation  r  (Å)"); axR.set_ylabel("energy  $U_{LJ}(r)$  (kcal/mol)")
axR.set_title("where the pair sits on the potential curve")

def animate(i):
    r = traj_r[i]
    pA.center = (-r/2, 0); pB.center = (+r/2, 0)
    dot.set_data([r], [lj(r)])
    txt.set_text(f"t     = {i*dt*TIME_FS/1000:5.2f} ps\n"
                 f"r     = {r:6.2f} A\n"
                 f"speed = {traj_v[i]:5.1f} A/ps")
    return pA, pB, dot, txt

anim = FuncAnimation(fig, animate, frames=frames, interval=60, blit=True)
plt.close(fig)
anim

---
## **14. Take-home messages**

**Lennard-Jones (van der Waals)**

* It is a **sum of two competing power laws**: a steep $+4\epsilon(\sigma/r)^{12}$ **repulsion**
  (Pauli exclusion, electron-cloud overlap) and a softer $-4\epsilon(\sigma/r)^{6}$ **attraction**
  (London dispersion). The $-6$ exponent is physically derived; the $-12$ is chosen for
  computational convenience, because $r^{-12}=\left(r^{-6}\right)^2$.
* Because the repulsion decays **twice as fast**, there is a window just outside contact where
  only the attraction survives — that window is the **well**. The two terms cancel exactly at
  $r=\sigma$ and balance in slope at $r=R_{min}=2^{1/6}\sigma$, where $U=-\epsilon$.
* $\epsilon$ scales the curve **vertically** (depth), $\sigma$ **horizontally** (size); the two
  parameters are essentially independent, and real atom types show it — oxygen is *smaller* than
  carbon yet **twice as sticky**.
* ⚠️ **Check the convention.** AMBER and CHARMM tabulate $R_{min}/2$, not $\sigma$, and
  $R_{min}=2^{1/6}\sigma$ is 12 % larger. Mixing the two up is a silent, classic error.
* The **force** $-dU/dr$ vanishes at $R_{min}$, diverges like $r^{-13}$ inside it — the reason MD
  needs a [soft core](LJ-ELEC_MD-SoftCore.ipynb) — and is strongest attractively at
  $1.24\,\sigma$, where for two carbons it is a mere 5 pN.

**The numbers matter, and they are small**

* For a C···C pair: $\sigma=3.40$ Å, $R_{min}=3.82$ Å, $\epsilon=0.109$ kcal/mol. Against
  $k_BT=0.596$ kcal/mol at 300 K, **one van der Waals contact is worth only $0.18\,k_BT$** — five
  times weaker than thermal noise, and the Boltzmann factor at contact is a feeble 1.20.
* So dispersion never binds anything *alone*: it works by **numbers**. A protein core or a
  membrane makes hundreds of these contacts at once. (In water the hydrophobic effect adds an
  extra, entropic driving force that this pair potential does not describe.)
* The **repulsive wall**, by contrast, is impenetrable at any accessible temperature.

**Coulomb (electrostatics)**

* A single $1/r$ with **no length scale and no minimum**: like charges repel at every distance,
  unlike charges attract at every distance and would collapse were it not for the Lennard-Jones
  wall.
* It is **far stronger at short range**. A $\pm1\,e$ ion pair at van der Waals contact is $-87$
  kcal/mol in vacuum — 800 times the C···C well, 146 $k_BT$. Even a $\pm0.1\,e$ partial charge is
  worth 8 van der Waals wells.
* The dielectric constant simply **divides** the curve, and that is what makes biology possible:
  the same ion pair is $-1.09$ kcal/mol $\approx1.8\,k_BT$ in water — about the strength quoted
  for a surface **salt bridge**, strong enough to matter and weak enough to break.

**The two together**

* **Range is the real difference**: $r^{-6}$ versus $r^{-1}$. At a standard 10 Å cutoff the van
  der Waals attraction has lost 99.7 % of its contact value — truncating it is harmless — while
  **38 %** of the Coulomb interaction is still there and gets chopped off. Real codes therefore
  use switching functions, reaction fields or Ewald/PME for electrostatics.
* With realistic partial charges the Coulomb term **reshapes the well**: in vacuum even
  $\pm0.1\,e$ deepens an unlike pair ninefold and destroys the minimum of a like pair altogether;
  in water the same charges are screened into a ~10 % correction and van der Waals runs the
  geometry again.
* Everything the minimisation, MD and Monte Carlo notebooks show — clustering of $+/-$ pairs,
  mutual avoidance of like charges, collisions that need a soft core — follows directly from these
  two curves.


---
## **15. Exercises**

The sections above did the work for you. These two exercises ask you to do it yourself: the first
has you **build a potential**, the second has you **use one to answer a physical question**.

> **How they work.** Each exercise starts from a stub containing the placeholder `TODO(r)`.
> Replace it with your own formula and re-run the cell. The cells are **self-checking**: they stay
> quiet until your function works, then fill themselves in with the numbers and plots you need.
> A worked solution is folded away under each exercise — try it yourself first.

---

### **Exercise 1 — a softer potential: 8-4 versus 12-6**

The 12-6 exponents are a convention, not a law of nature (§1). A general **n-m** potential is

$$U_{nm}(r)=C_{nm}\,\epsilon\left[\left(\frac{\sigma}{r}\right)^{n}-\left(\frac{\sigma}{r}\right)^{m}\right],
\qquad C_{nm}=\frac{n}{n-m}\left(\frac{n}{m}\right)^{\frac{m}{n-m}}$$

The prefactor $C_{nm}$ is chosen so that — whatever $n$ and $m$ — the curve still crosses zero at
$r=\sigma$ and its minimum is still exactly $-\epsilon$. That is what makes a fair comparison
possible: change the exponents and *nothing else* changes.

**(a)** Work out $C_{nm}$ for the familiar **12-6** case; you should recover the 4 we have used all
along. Now do the same for **8-4**. (The answer is pleasantly simple.)

**(b)** Code `lj_84_repulsion`, `lj_84_attraction` and `lj_84` in the cell below, by analogy with
the 12-6 functions of §5.

**(c)** Show — on paper, then check numerically — that the 8-4 minimum sits at
$R_{min}=2^{1/4}\sigma$ rather than $2^{1/6}\sigma$.

**(d)** Plot the two potentials together, with the **same** $\epsilon$ and $\sigma$, and compare
them quantitatively:

* what does it cost to squeeze the pair to $0.9\,\sigma$, $0.8\,\sigma$ and $0.7\,\sigma$ under each?
* how wide is each well, measured at half its depth?
* how much attraction is left at $2\,\sigma$, $3\,\sigma$, $5\,\sigma$ and $10\,\sigma$? Can you see
  the pattern in the ratios? (Compare them with $(r/\sigma)^2$.)

**(e)** A question with no code attached: **why would anyone deliberately use a softer potential?**
Think about what a $r^{-12}$ wall does to a structure that is nearly — but not exactly — right.

In [ ]:
# ---------- Exercise 1: complete the functions marked TODO ----------
TODO = lambda r: np.nan*np.asarray(r, dtype=float)      # placeholder — delete when you fill in

def solved(f):
    '''True once f returns real numbers instead of the TODO placeholder.'''
    try:
        return not np.isnan(np.asarray(f(4.0), dtype=float)).any()
    except Exception:
        return False


def lj_nm_prefactor(n, m):
    '''(a)  C(n,m) = n/(n-m) * (n/m)**(m/(n-m)).   C(12,6) must come out as 4.'''
    return np.nan                                        # <-- your code here

def lj_84_repulsion(r, epsilon=Epsilon, sigma=Sigma):
    '''(b)  the  +C eps (sigma/r)^8  term.'''
    return TODO(r)                                       # <-- your code here

def lj_84_attraction(r, epsilon=Epsilon, sigma=Sigma):
    '''(b)  the  -C eps (sigma/r)^4  term.'''
    return TODO(r)                                       # <-- your code here

def lj_84(r, epsilon=Epsilon, sigma=Sigma):
    return lj_84_repulsion(r, epsilon, sigma) + lj_84_attraction(r, epsilon, sigma)


# ---------- self-check ----------
print("(a) prefactors")
for n, m in ((12, 6), (8, 4)):
    c = lj_nm_prefactor(n, m)
    verdict = "" if np.isnan(c) else ("OK" if abs(c - 4) < 1e-9 else "hmm, expected 4")
    print(f"    C({n:2d},{m}) = {c!s:<22} {verdict}")

print("\n(b,c) landmarks of your 8-4 curve")
if solved(lj_84):
    g  = np.linspace(0.9*Sigma, 4.0*Sigma, 400_001)
    u  = lj_84(g)
    i  = int(np.argmin(u))
    ok = lambda got, want, tol: "OK" if abs(got - want) < tol else f"<- expected {want:.4f}"
    print(f"    U(sigma)         = {float(lj_84(Sigma)):+.3e}     {ok(float(lj_84(Sigma)), 0.0, 1e-9)}")
    print(f"    minimum at r     = {g[i]:.4f} A        {ok(g[i], 2**0.25*Sigma, 1e-3)}")
    print(f"    depth at minimum = {float(u[i]):+.4f}          {ok(float(u[i]), -Epsilon, 1e-6)}")
    print(f"\n    for comparison, 12-6 has its minimum at {Rmin:.4f} A, same depth {-Epsilon:+.4f}")
else:
    print("    -> not yet: replace the TODO(r) placeholders above and re-run this cell.")

The cell below draws the comparison asked for in part **(d)**. The 12-6 curve is always
shown; your 8-4 curve appears as soon as the functions above work. Both are plotted in units of
$\sigma$ and $\epsilon$, so the picture applies to *any* atom pair, not just carbon.

In [ ]:
# Visualise the 12-6 and 8-4 potentials together
C_126, C_84 = "#6a3d9a", "#1b8a5a"      # purple = 12-6, green = 8-4

fig, (axW, axL) = plt.subplots(1, 2, figsize=(12.5, 4.6))
x = np.linspace(0.85, 3.2, 700)          # r / sigma
r = x*Sigma

# ---- left: the two wells side by side ----
axW.plot(x, lj(r)/Epsilon, color=C_126, lw=2.3, label="12-6")
axW.axhline(0, color=C_REF, lw=0.8); axW.axhline(-1, color=C_REF, ls=":", lw=0.9)
axW.axvline(2**(1/6), color=C_126, ls=":", lw=0.9)
if solved(lj_84):
    axW.plot(x, lj_84(r)/Epsilon, color=C_84, lw=2.3, label="8-4  (softer)")
    axW.axvline(2**(1/4), color=C_84, ls=":", lw=0.9)
axW.annotate(r"depth $-\epsilon$ for both", xy=(2.6, -1), xytext=(1.95, -0.80),
             color="#444444", fontsize=9)
axW.set_xlim(0.85, 3.2); axW.set_ylim(-1.5, 3.0)
axW.set_xlabel(r"separation  $r/\sigma$"); axW.set_ylabel(r"energy  $U/\epsilon$")
axW.set_title("same depth, same zero crossing — different shape")
axW.legend(loc="upper right")

# ---- right: the long-range tail, log-log ----
xt = np.linspace(1.2, 12.0, 700); rt = xt*Sigma
axL.loglog(xt, -lj(rt)/Epsilon, color=C_126, lw=2.3, label=r"12-6   $\propto r^{-6}$")
if solved(lj_84):
    axL.loglog(xt, -lj_84(rt)/Epsilon, color=C_84, lw=2.3, label=r"8-4    $\propto r^{-4}$")
axL.set_xlabel(r"separation  $r/\sigma$  (log)")
axL.set_ylabel(r"attraction  $|U|/\epsilon$  (log)")
axL.set_title("the 8-4 tail reaches much further")
axL.set_xticks([2, 3, 5, 8, 12]); axL.set_xticklabels(["2", "3", "5", "8", "12"])
axL.minorticks_off(); axL.legend(loc="upper right")

plt.tight_layout(); plt.show()

if not solved(lj_84):
    print("Complete the functions in the previous cell to add the 8-4 curve.")
else:
    def half_depth_width(U):
        g = np.linspace(0.95*Sigma, 8*Sigma, 400_001)
        inside = g[U(g) < -Epsilon/2]
        return inside.min(), inside.max(), inside.max() - inside.min()

    print("cost of squeezing the pair (kcal/mol)")
    print(f"{'r':>10} {'12-6':>10} {'8-4':>10} {'ratio':>8}")
    for f in (0.9, 0.8, 0.7):
        a, b = float(lj(f*Sigma)), float(lj_84(f*Sigma))
        print(f"{f:>7.1f}*s {a:10.3f} {b:10.3f} {a/b:8.2f}")

    print("\nwidth of the well at half depth (A)")
    for name, U in (("12-6", lj), ("8-4 ", lj_84)):
        lo, hi, w = half_depth_width(U)
        print(f"  {name}: {lo:.3f} -> {hi:.3f}   width {w:.3f}")

    print("\nattraction term remaining (kcal/mol)")
    print(f"{'r':>10} {'12-6':>13} {'8-4':>13} {'ratio':>8} {'(r/sigma)^2':>13}")
    for f in (2, 3, 5, 10):
        a, b = float(lj_attraction(f*Sigma)), float(lj_84_attraction(f*Sigma))
        print(f"{f:>7d}*s {a:13.3e} {b:13.3e} {b/a:8.1f} {f**2:13d}")

<details>
<summary><b>Exercise 1 — worked solution and expected answers</b></summary>

**(a)** $C_{12,6}=\dfrac{12}{6}\left(\dfrac{12}{6}\right)^{6/6}=2\times2=4$, and
$C_{8,4}=\dfrac{8}{4}\left(\dfrac{8}{4}\right)^{4/4}=2\times2=4$ — **also exactly 4**, so the 8-4
potential is written with the same familiar prefactor.

**(b)**

```python
def lj_nm_prefactor(n, m):
    return n/(n - m) * (n/m)**(m/(n - m))

def lj_84_repulsion(r, epsilon=Epsilon, sigma=Sigma):
    return 4*epsilon * (sigma/np.asarray(r, dtype=float))**8

def lj_84_attraction(r, epsilon=Epsilon, sigma=Sigma):
    return -4*epsilon * (sigma/np.asarray(r, dtype=float))**4
```

**(c)** Setting $\mathrm{d}U/\mathrm{d}r=0$ gives $8(\sigma/r)^{8}=4(\sigma/r)^{4}$, i.e.
$(\sigma/r)^{4}=\tfrac12$, so $R_{min}=2^{1/4}\sigma=1.189\,\sigma=4.043$ Å — against
$2^{1/6}\sigma=1.122\,\sigma=3.816$ Å for 12-6. Substituting back,
$U=4\epsilon(\tfrac14-\tfrac12)=-\epsilon$: the depth is unchanged, as promised.

**(d)** With $\sigma=3.40$ Å and $\epsilon=0.1094$ kcal/mol:

| | 12-6 | 8-4 | ratio |
|---|---|---|---|
| squeeze to $0.9\,\sigma$ | 0.73 | 0.35 | 2.1 |
| squeeze to $0.8\,\sigma$ | 4.70 | 1.54 | 3.1 |
| squeeze to $0.7\,\sigma$ | 27.9 | 5.77 | 4.8 |
| width of well at half depth | 1.19 Å | 1.96 Å | 0.6 |
| attraction at $3\,\sigma$ | $-6.0\times10^{-4}$ | $-5.4\times10^{-3}$ | 9 |
| attraction at $10\,\sigma$ | $-4.4\times10^{-7}$ | $-4.4\times10^{-5}$ | 100 |

The 8-4 is softer in **three** distinct ways: its wall is several times cheaper to climb, its well
is about **60 % wider**, and its tail reaches much further. The attraction ratio is exactly
$(r/\sigma)^{2}$ — 4, 9, 25, 100 at $2,3,5,10\,\sigma$ — because $r^{-4}$ decays two powers more
slowly than $r^{-6}$.

**(e)** A $r^{-12}$ wall is unforgiving: an atom placed 0.5 Å too close costs *kilocalories*, which
in a simulation becomes a huge force and a violent kick. Whenever the input geometry is only
approximately right — **protein–protein docking**, **coarse-grained** models where one bead stands
for several atoms, or the early stages of a refinement — that sensitivity is a liability rather
than realism. A softer potential lets near-misses relax instead of exploding, smooths the energy
landscape so that minimisers and samplers get trapped less often, and tolerates larger time steps.
The price is less realistic close packing: a soft potential will happily let atoms overlap a
little, so it is typically used for the early, approximate stages and swapped for the true 12-6 at
the end. (The [soft core](LJ-ELEC_MD-SoftCore.ipynb) of the MD notebook is a cruder version of the
same idea — it caps the wall rather than reshaping it.)

</details>



---
### **Exercise 2 — how much screening balances electrostatics against van der Waals?**

Section 13 showed the two extremes for a $\pm0.1\,e$ partial-charge pair: in **vacuum** it deepens
the well ninefold, in **water** it is a 10 % correction. Somewhere in between there must be a
dielectric constant at which the two interactions are **exactly balanced**.

Take balance to mean: *at the van der Waals contact distance $R_{min}$, the magnitude of the
Coulomb energy equals the depth of the van der Waals well,*

$$\left|\frac{332.06\;q_a q_b}{\varepsilon_r\,R_{min}}\right| \;=\; \epsilon$$

**(a)** Solve that for $\varepsilon_r$ and code it as `dielectric_for_balance(q)` below.

**(b)** Evaluate it for $q=\pm0.1$, $\pm0.2$, $\pm0.5$ and $\pm1.0\,e$. How does the answer scale
with the charge?

**(c)** Look each answer up in the table of measured dielectric constants below. **Which solvent
balances a $\pm0.1\,e$ pair?** And what does the answer for a full $\pm1\,e$ ion pair tell you about
whether *any* solvent can reduce electrostatics to van der Waals strength?

**(d)** Check yourself: plot the total potential at your balancing $\varepsilon_r$. If the answer is
right the well should come out almost exactly **twice** as deep as the uncharged one — why?

**Static dielectric constants at room temperature**

| solvent | $\varepsilon_r$ | | solvent | $\varepsilon_r$ |
|---|---|---|---|---|
| vacuum | 1.0 | | 1-octanol | 10.3 |
| hexane | 1.9 | | liquid ammonia | 17 |
| benzene / toluene | 2.3 | | acetone | 20.7 |
| diethyl ether | 4.3 | | ethanol | 24.5 |
| chloroform | 4.8 | | methanol | 32.7 |
| ethyl acetate | 6.0 | | glycerol | 42.5 |
| tetrahydrofuran (THF) | 7.6 | | water | 80.1 |
| dichloromethane | 8.9 | | formamide | 111 |

*(For reference, the interior of a folded protein is usually modelled with
$\varepsilon_r\approx2\!-\!4$.)*

In [ ]:
# ---------- Exercise 2: complete the function marked TODO ----------
def dielectric_for_balance(q, r=None, epsilon=Epsilon):
    '''(a)  eps_r at which |U_Coul| of a +q/-q pair at separation r equals the
            van der Waals well depth.  Default r: the contact distance Rmin.'''
    if r is None:
        r = Rmin
    return np.nan                                        # <-- your code here


SOLVENTS = {"vacuum": 1.0, "hexane": 1.9, "benzene/toluene": 2.3, "diethyl ether": 4.3,
            "chloroform": 4.8, "ethyl acetate": 6.0, "THF": 7.6, "dichloromethane": 8.9,
            "1-octanol": 10.3, "liquid ammonia": 17.0, "acetone": 20.7, "ethanol": 24.5,
            "methanol": 32.7, "glycerol": 42.5, "water": 80.1, "formamide": 111.0}

if np.isnan(dielectric_for_balance(0.1)):
    print("Not yet: replace the placeholder in dielectric_for_balance() and re-run this cell.")
else:
    print(f"(b,c) balancing dielectric constant   (contact distance {Rmin:.2f} A, "
          f"vdW well {Epsilon:.4f} kcal/mol)\n")
    print(f"{'q (e)':>7} {'eps_r needed':>14}   closest real solvent")
    for q in (0.1, 0.2, 0.5, 1.0):
        d = float(dielectric_for_balance(q))
        if d > max(SOLVENTS.values()):
            match = "-- no solvent comes close --"
        else:
            name = min(SOLVENTS, key=lambda s: abs(SOLVENTS[s] - d))
            match = f"{name} ({SOLVENTS[name]})"
        print(f"{q:>7.2f} {d:14.1f}   {match}")
    print("\n    eps_r scales as q^2: doubling the charge needs 4x the screening.")

    # ---------- (d) verification ----------
    q_ex  = 0.1
    d_bal = float(dielectric_for_balance(q_ex))
    rr = np.linspace(0.88*Sigma, 3.0*Sigma, 800)

    fig, ax = plt.subplots(figsize=(8.4, 4.6))
    ax.plot(rr, lj(rr), color=C_REF, ls="--", lw=1.6, label="Lennard-Jones only (uncharged)")
    ax.plot(rr, lj(rr) + coulomb(rr, +q_ex, -q_ex, d_bal), color=C_ATT, lw=2.3,
            label=rf"LJ + Coulomb at $\varepsilon_r$ = {d_bal:.1f}")
    ax.axhline(0, color=C_REF, lw=0.8)
    ax.axhline(-Epsilon,   color=C_REF, ls=":", lw=0.9)
    ax.axhline(-2*Epsilon, color=C_REF, ls=":", lw=0.9)
    ax.annotate(r"$-\epsilon$",  xy=(2.9*Sigma, -Epsilon),
                xytext=(2.45*Sigma, -Epsilon + 0.013), color="#444444", fontsize=9)
    ax.annotate(r"$-2\epsilon$", xy=(2.9*Sigma, -2*Epsilon),
                xytext=(2.45*Sigma, -2*Epsilon + 0.013), color="#444444", fontsize=9)
    ax.set_ylim(-0.30, 0.22)
    ax.set_xlabel("pair separation  r  (Å)"); ax.set_ylabel("total pair energy  (kcal/mol)")
    ax.set_title(rf"(d) at the balancing $\varepsilon_r$ the well is twice as deep  ($q=\pm{q_ex}\,e$)")
    ax.legend(loc="upper right")
    plt.tight_layout(); plt.show()

    g = np.linspace(0.85*Sigma, 4*Sigma, 400_001)
    u = lj(g) + coulomb(g, +q_ex, -q_ex, d_bal)
    i = int(np.argmin(u))
    print(f"    uncharged minimum : r = {Rmin:.3f} A,  U = {-Epsilon:+.4f} kcal/mol")
    print(f"    balanced  minimum : r = {g[i]:.3f} A,  U = {u[i]:+.4f} kcal/mol "
          f"  ({u[i]/-Epsilon:.2f}x deeper)")

<details>
<summary><b>Exercise 2 — worked solution and expected answers</b></summary>

**(a)** Rearranging $\left|332.06\,q_aq_b/(\varepsilon_r R_{min})\right|=\epsilon$ with
$q_a=-q_b=q$:

```python
def dielectric_for_balance(q, r=None, epsilon=Epsilon):
    if r is None:
        r = Rmin
    return COULOMB_K * q*q / (r * epsilon)
```

**(b)**

| $q$ | $\varepsilon_r$ needed | closest real solvent |
|---|---|---|
| $\pm0.1\,e$ | **8.0** | THF (7.6) / dichloromethane (8.9) |
| $\pm0.2\,e$ | 31.8 | methanol (32.7) |
| $\pm0.5\,e$ | 198.8 | none — nearly twice formamide |
| $\pm1.0\,e$ | 795.3 | none — ten times water |

Because the Coulomb energy goes as $q_aq_b=q^{2}$, so does the screening needed: **doubling the
charge requires four times the dielectric constant.**

**(c)** A $\pm0.1\,e$ pair is balanced at $\varepsilon_r\approx8$ — roughly **tetrahydrofuran or
dichloromethane**, a moderately polar organic solvent, and also about the value often used for a
partly buried site in a protein.

The full ion pair is the more interesting answer: it would need $\varepsilon_r\approx800$, **ten
times that of water**, and no liquid comes close. That is the real lesson — *no solvent can screen
a full $\pm1\,e$ contact ion pair down to van der Waals strength.* Even in water it stays about ten
times the van der Waals well (§11). Whenever full charges are present, electrostatics sets the
structure and dispersion merely fine-tunes it.

**(d)** At the balancing $\varepsilon_r$ the Coulomb term contributes exactly $-\epsilon$ at
$R_{min}$, on top of the van der Waals $-\epsilon$, so the well bottom lands near $-2\epsilon$. It
is not *exactly* $-2\epsilon$, and the minimum shifts slightly inwards (to about 3.7 Å), because
the added $1/r$ attraction is still growing as $r$ decreases: the sum bottoms out at 3.77 Å,
a little closer in than either term alone.

</details>



---

## **Going further**

Open-ended, no scaffolding provided:

* **Other exponents.** The **9-6** potential is used by force fields such as COMPASS and PCFF.
  Work out $C_{9,6}$ and the position of its minimum, and place its curve between the 12-6 and the
  8-4.
* **A temperature scale for stickiness.** At what temperature would the C···C well be worth a full
  $k_BT$? (Section 10 has everything you need; the answer is surprisingly cold.)
* **Does the atom matter?** Redo Exercise 2 for a sulphur pair ($\epsilon=0.25$ kcal/mol,
  $\sigma=3.56$ Å) instead of carbon. Does a *stickier* atom need more screening or less — and why?
* **Does the distance matter?** Redo Exercise 2 at 5 Å and at 8 Å instead of at contact. Which
  interaction fades faster as the pair separates, and does that change which solvent balances them?